# Copper forecasting — NON-SA complete fair benchmark (17/11/2026)

**Protocol cố định:** Dự báo Copper(t) từ dữ liệu **tối đa tới t−1** (horizon 1). Huấn luyện lại từ CSV, **không có SA**, không chọn mô hình dựa vào test. Tất cả mô hình được đánh giá trên **cùng target timestamps**.

**Full factorial:** 3 feature sets (Copper-only / Copper+Paper3 / Copper+Legacy14) × 3 strategies (expanding / rolling756 / recency-weighted) × 4 splits (80/20, 90/10, 92.5/7.5, 95/5) × **9 fitted models** = **324 fits**, cộng Naive cho mỗi tổ hợp. Nhóm neural gồm LSTM, GRU, CNN-LSTM, CNN-GRU ở cả hai objective **direct** và **residual**; thêm Residual Ridge; **không chạy XGBoost**. Một seed cho full factorial; nghiên cứu độ ổn định 5 seeds ở walk-forward Residual-GRU riêng.

**Đặc biệt:** (1) Kiểm tra Copper-only và giá trị gia tăng của biến ngoại sinh; (2) kiểm tra residual prediction có thực sự học được delta hơn 0 hay gần như lặp Naive. Ghi rõ kết quả thua Naive nếu có.

**Chi phí chạy:** FULL_MODE nhiều trăm fits, cần GPU và có thể vượt giới hạn một phiên Kaggle. Notebook có **resume CSV** theo khóa cấu hình; chạy lại cell train sẽ bỏ qua jobs hoàn thành. `QUICK_MODE` chạy logic toàn bộ tổ hợp với epochs thấp, **không dùng để công bố kết quả cuối**. `FULL_MODE` dùng cho báo cáo. Không tự cắt bớt tổ hợp trong FULL.

## 1. Configuration & environment

In [ ]:
from pathlib import Path
import os, json, gc, time, zipfile, warnings, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf

MODE = 'FULL_MODE'  # QUICK_MODE để thử notebook; FULL_MODE để báo cáo
LOOKBACK = 22
HORIZON = 1   # KHÓA: price(t) dự đoán từ observations <= t-1
RATIOS = (0.80, 0.90, 0.925, 0.95)
FEATURE_SETS = {
    'copper_only': ['Copper'],
    'copper_paper3': ['Copper','WTI','Gold','Silver'],
    'copper_legacy14': ['Copper','Aluminium','Tin','Silver','Gold']
}
SCHEMES = ('expanding','rolling_756','recency_weighted')
NN_ARCHES = ('lstm','gru','cnn_lstm','cnn_gru')
MODEL_NAMES = [f'{a}_{objective}' for objective in ('direct','residual') for a in NN_ARCHES] + ['ridge_residual']
SEED = 42
EPOCHS = 65 if MODE == 'FULL_MODE' else 3
PATIENCE = 9 if MODE == 'FULL_MODE' else 2
BATCH_SIZE = 64
WF_START_YEAR = 2018
WF_STEP = 63 if MODE == 'FULL_MODE' else 252
WF_SEEDS = (41,42,43,44,45) if MODE == 'FULL_MODE' else (42,43)
WF_FEATURE_SET = 'copper_paper3'  # a priori, KHÔNG chọn dựa vào test
WF_EPOCHS = 45 if MODE == 'FULL_MODE' else 3
WF_PATIENCE = 7 if MODE == 'FULL_MODE' else 2
RUN_FULL_FACTORIAL = True
RUN_WALK_FORWARD = True

CANDIDATES = [
    Path('/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
    Path('/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
    Path('data/processed/dataset_9_commodities_daily.csv'),
    Path('dataset_9_commodities_daily.csv')
]
OUT = Path('/kaggle/working/copper_non_sa_fair_v20') if Path('/kaggle/working').exists() else Path('copper_non_sa_fair_v20')
OUT.mkdir(parents=True, exist_ok=True)

# Mở lại notebook: giữ kết quả cùng config; nếu đổi MODE/lookback, tự dùng thư mục khác
RUN_KEY = f'{MODE}_lb{LOOKBACK}_h{HORIZON}_seed{SEED}_ep{EPOCHS}'
OUT = OUT / RUN_KEY
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
print('═'*72)
print('COPPER FORECASTING | NON-SA FAIR BENCHMARK V20')
print('═'*72)
print(f'Mode: {MODE} | Forecast: t-1 → t | Lookback: {LOOKBACK} phiên')
print(f'Splits: {[f"{r*100:g}/{(1-r)*100:g}" for r in RATIOS]}')
print(f'Features: {list(FEATURE_SETS)}')
print(f'Models: {len(MODEL_NAMES)} fitted + Naive | Schemes: {list(SCHEMES)}')
print(f'Full grid: {len(RATIOS)*len(FEATURE_SETS)*len(SCHEMES)*len(MODEL_NAMES)} fits + {len(RATIOS)*len(FEATURE_SETS)*len(SCHEMES)} Naive')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Checkpoint: {OUT}')
print('═'*72)

## 2. Data ingest, timestamp audit, quality checks

In [ ]:
data_path = next((p for p in CANDIDATES if p.is_file()), None)
if data_path is None and Path('/kaggle/input').exists():
    matches = list(Path('/kaggle/input').rglob('dataset_9_commodities_daily.csv'))
    if len(matches)==1: data_path=matches[0]
if data_path is None:
    raise FileNotFoundError('Gắn dataset_9_commodities_daily.csv trên Kaggle; không dùng dữ liệu mô phỏng để report.')
raw = pd.read_csv(data_path)
required = set(['Date'] + list(dict.fromkeys(sum(FEATURE_SETS.values(),[]))))
if not required.issubset(raw.columns): raise ValueError('Missing required columns: '+repr(sorted(required-set(raw.columns))))
raw['Date'] = pd.to_datetime(raw['Date'], errors='coerce')
if raw['Date'].isna().any(): raise ValueError('Date không hợp lệ')
if raw['Date'].duplicated().any(): raise ValueError('Duplicate Date: phải kiểm tra trước khi train')
raw=raw.sort_values('Date').reset_index(drop=True)
for c in required-{'Date'}:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')
quality = pd.DataFrame({'column':list(raw.columns), 'dtype':raw.dtypes.astype(str).values,
    'missing':raw.isna().sum().values, 'unique':raw.nunique(dropna=True).values})
quality.to_csv(OUT/'quality.csv',index=False)
nonpos = {c:int((raw[c]<=0).sum()) for c in required-{'Date'}}
(OUT/'nonpositive_counts.json').write_text(json.dumps(nonpos,indent=2))
# Chỉ loại thiếu trong union các feature được dùng; không lọc theo cột bên ngoài ma trận
union_cols = list(dict.fromkeys(sum(FEATURE_SETS.values(),[])))
data=raw.dropna(subset=union_cols).reset_index(drop=True)
assert len(data)>1000 and data.Date.is_monotonic_increasing
assert all((data[c]>0).all() for c in union_cols), 'Giá <=0: cần audit nguồn dữ liệu trước khi tính returns.'
removed= len(raw)-len(data)
# Bảo vệ resume: không được trộn kết quả từ 2 dataset khác nhau trong cùng OUT.
dataset_fingerprint=hashlib.sha256(Path(data_path).read_bytes()).hexdigest()
signature_file=OUT/'dataset_fingerprint.txt'
if signature_file.exists() and signature_file.read_text().strip()!=dataset_fingerprint:
    raise RuntimeError('Dataset đã thay đổi nhưng output cũ vẫn còn. Xóa/đổi tên OUT trước khi resume.')
signature_file.write_text(dataset_fingerprint)
print('File:',data_path,'| rows:',len(data),'| removed due to missing:',removed,
      '| start:',data.Date.iloc[0].date(),'| end:',data.Date.iloc[-1].date())
display(quality)
if removed: print('WARNING: Loại dòng missing tạo chuỗi phiên giao dịch liền theo các ngày CÒN LẠI; kiểm tra nguyên nhân missing!')
# Dữ liệu CSV được giả định là giá daily close, không xác minh được publication-time từ CSV.
# Chỉ sử dụng các giá trị daily tại <=t-1; với sản phẩm và múi giờ khác nhau cần audit giờ công bố.
assert HORIZON == 1

## 3. EDA: levels, returns, volatility, Spearman, lagged correlations (train-only correlation)

EDA toàn chuỗi dùng **mô tả**, không chọn features qua test. Tương quan dùng train ban đầu để lập giả thuyết; kết luận không dựa vào tương quan full series.

In [ ]:
EDA_COLS = ['Copper','WTI','Gold','Silver','Aluminium','Tin']
returns = data[EDA_COLS].pct_change(fill_method=None).replace([np.inf,-np.inf],np.nan)
fig,axs = plt.subplots(3,1,figsize=(15,13),sharex=True)
levels=data[EDA_COLS]
normalized=(levels-levels.iloc[:int(len(data)*.8)].min()) / (levels.iloc[:int(len(data)*.8)].max()-levels.iloc[:int(len(data)*.8)].min())
for c in EDA_COLS: axs[0].plot(data.Date,normalized[c],label=c,linewidth=1.1)
axs[0].set_title('Relative price level; visual scaling uses initial 80% only');axs[0].legend(ncol=3)
axs[1].plot(data.Date,returns['Copper'],lw=.8);axs[1].set_title('Copper daily return')
axs[2].plot(data.Date,returns['Copper'].rolling(22).std(),lw=1);axs[2].set_title('Copper rolling volatility (22 sessions)')
for a in axs:a.grid(alpha=.2)
plt.tight_layout();plt.savefig(OUT/'eda_levels_returns_volatility.png',dpi=150);plt.show()
training_eda = data.iloc[:int(len(data)*0.90*0.90)]
sp = training_eda[EDA_COLS].corr(method='spearman')
sp.to_csv(OUT/'spearman_training_only.csv')
display(sp.round(3))
rows=[]
trret=training_eda[EDA_COLS].pct_change(fill_method=None)
for c in EDA_COLS[1:]:
  for lag in [1,2,5,10,22,50]:
    pair=pd.DataFrame({'y':trret.Copper, 'x':trret[c].shift(lag)}).dropna()
    rows.append({'feature':c,'lag':lag,'n':len(pair),'pearson':pair.y.corr(pair.x),'spearman':pair.y.corr(pair.x,method='spearman')})
lagged=pd.DataFrame(rows);lagged.to_csv(OUT/'lagged_correlations_training_only.csv',index=False)
for method in ['pearson','spearman']:
  mat=lagged.pivot(index='feature',columns='lag',values=method)
  fig,ax=plt.subplots(figsize=(9,4));im=ax.imshow(mat,vmin=-1,vmax=1,cmap='coolwarm',aspect='auto')
  ax.set_xticks(range(len(mat.columns)),mat.columns);ax.set_yticks(range(len(mat.index)),mat.index)
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):ax.text(j,i,f'{mat.iloc[i,j]:.2f}',ha='center',va='center',fontsize=9)
  ax.set_title(f'Train-only lagged {method}: feature(t-lag) vs Copper(t) returns')
  fig.colorbar(im);plt.tight_layout();plt.savefig(OUT/f'eda_lagged_{method}.png',dpi=150);plt.show()

## 4. Leak-safe splitting and scaling

- Split **the target indices** chronologically. Initial training is `[fit_start, val_start)`, validation `[val_start, test_start)`, test `[test_start, n)`.
- Expanding: `fit_start=0`; Rolling756: `fit_start=max(0,val_start−756)`; Recency: toàn bộ train + exponential sample weights.
- Scaler `fit` **chỉ trên initial train** trước validation; validation/test luôn transform bằng scaler đó.
- Tất cả windows lấy `t−22..t−1`. Nhánh residual: label `y_scaled(t)-y_scaled(t-1)`; cuối cùng cộng `Copper_scaled(t-1)` và inverse về raw. Nhánh direct dự báo `Copper_scaled(t)`.
- **Dữ liệu gần ranh giới:** validation/test có thể sử dụng lịch sử train đã quan sát; không sử dụng nhãn tương lai.
- Feature sets bao gồm Copper nên các mô hình có cùng quyền truy cập autoregressive information. Naive sử dụng Copper(t−1).

In [ ]:
def make_context(frame, ratio, scheme, feature_set, val_fraction=.10):
    n=len(frame); split=int(n*ratio); val_start=int(split*(1-val_fraction))
    fit_start=0 if scheme!='rolling_756' else max(0,val_start-756)
    cols=FEATURE_SETS[feature_set]
    assert LOOKBACK+25<val_start<split<n and split< n
    assert (val_start-fit_start)>LOOKBACK+10
    xsc=MinMaxScaler().fit(frame.loc[fit_start:val_start-1, cols])
    ysc=MinMaxScaler().fit(frame.loc[fit_start:val_start-1,['Copper']])
    x=xsc.transform(frame[cols]).astype('float32')
    y=ysc.transform(frame[['Copper']]).ravel().astype('float32')
    # prev at t-1 is same Copper scalers, regardless input group
    prev=np.r_[np.nan,y[:-1]]
    def block(start,end,restrict_train=False):
        indices=np.arange(start,end)
        indices=indices[indices>=LOOKBACK]
        if restrict_train: indices=indices[indices-LOOKBACK>=fit_start]
        X=np.stack([x[t-LOOKBACK:t] for t in indices]).astype('float32')
        return {'idx':indices, 'X':X, 'y':y[indices], 'prev':prev[indices].astype('float32'),
                'date':frame.Date.iloc[indices].to_numpy()}
    tr=block(fit_start,val_start,True);va=block(val_start,split);te=block(split,n)
    for block_name,b in [('train',tr),('val',va),('test',te)]:
        assert np.isfinite(b['X']).all() and np.isfinite(b['y']).all() and np.isfinite(b['prev']).all(),block_name
        assert len(b['X'])==len(b['idx']) and b['X'].shape[1:]==(LOOKBACK,len(cols))
        assert (b['idx']-1< b['idx']).all()
        # exact assertion: t-1 Copper equals last input Copper, same scaler
        assert np.allclose(b['X'][:,-1,0],b['prev'],atol=2e-6,rtol=1e-6)
    assert tr['idx'][-1]<va['idx'][0] and va['idx'][-1]<te['idx'][0]
    assert xsc.n_samples_seen_==val_start-fit_start
    return {'train':tr,'val':va,'test':te,'xsc':xsc,'ysc':ysc,'features':cols,'feature_set':feature_set,
            'ratio':float(ratio),'scheme':scheme,'fit_start':fit_start,'val_start':val_start,'test_start':split}

for ratio in RATIOS:
    audits=[]
    for feat in FEATURE_SETS:
      for scheme in SCHEMES:
        ctx=make_context(data,ratio,scheme,feat)
        audits.append({'ratio':ratio,'features':feat,'scheme':scheme,'train':len(ctx['train']['idx']),
                       'val':len(ctx['val']['idx']),'test':len(ctx['test']['idx']),
                       'test_start':str(pd.Timestamp(ctx['test']['date'][0]).date()),
                       'scaler_fit_end':str(data.Date.iloc[ctx['val_start']-1].date()),
                       'input_last_day_test_first':str(data.Date.iloc[ctx['test']['idx'][0]-1].date())})
    # all feature sets & schemes must use exactly same test labels and dates
    anchor=make_context(data,ratio,'expanding','copper_only')['test']
    for feat in FEATURE_SETS:
      for scheme in SCHEMES:
        t=make_context(data,ratio,scheme,feat)['test']
        assert np.array_equal(t['idx'],anchor['idx'])
    if ratio==RATIOS[0]: display(pd.DataFrame(audits))
print('PASS: scaler fit before validation, all models same dates, Copper(t-1) exact, no target t in input.')

## 5. Architecture builders — SAME capacity per paired direct/residual

For each architecture, direct and residual use exactly the same network; only the training target differs. CNN is causal Conv1D; no SA.

In [ ]:
def build_net(arch, shape):
    inp=tf.keras.Input(shape=shape)
    z=inp
    if arch in ('cnn_lstm','cnn_gru'):
        z=tf.keras.layers.Conv1D(32,3,padding='causal',activation='relu')(z)
        z=tf.keras.layers.MaxPooling1D(2)(z)
    if arch in ('lstm','cnn_lstm'):
        n1,n2=(64,32) if arch=='lstm' else (48,24)
        z=tf.keras.layers.LSTM(n1,return_sequences=True)(z)
        z=tf.keras.layers.Dropout(.2)(z)
        z=tf.keras.layers.LSTM(n2)(z)
    elif arch in ('gru','cnn_gru'):
        n1,n2=(64,32) if arch=='gru' else (48,24)
        z=tf.keras.layers.GRU(n1,return_sequences=True)(z)
        z=tf.keras.layers.Dropout(.2)(z)
        z=tf.keras.layers.GRU(n2)(z)
    else: raise ValueError(arch)
    z=tf.keras.layers.Dropout(.2)(z)
    z=tf.keras.layers.Dense(16,activation='relu')(z)
    out=tf.keras.layers.Dense(1,activation='linear')(z)
    model=tf.keras.Model(inp,out,name=arch)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse')
    return model

def sample_weights(length, scheme, half_life=252):
    if scheme!='recency_weighted': return None
    ages=np.arange(length-1,-1,-1,dtype='float64')
    w=np.exp(-np.log(2)*ages/half_life)
    return (w/w.mean()).astype('float32')

def evaluate(ctx, model_name, p_val, p_test, seed=SEED, extras=None):
    ysc=ctx['ysc'];va=ctx['val'];te=ctx['test']
    def original(z): return ysc.inverse_transform(np.asarray(z).reshape(-1,1)).ravel()
    y_true=original(te['y']);pred=original(p_test);base=original(te['prev'])
    delta_pred=pred-base;delta_true=y_true-base
    mae=float(mean_absolute_error(y_true,pred));base_mae=float(mean_absolute_error(y_true,base))
    row={'model':model_name,'feature_set':ctx['feature_set'],'scheme':ctx['scheme'],
         'ratio':ctx['ratio'],'seed':int(seed),'n_test':len(y_true),'mae':mae,'rmse':float(np.sqrt(mean_squared_error(y_true,pred))),
         'mse':float(mean_squared_error(y_true,pred)), 'mae_naive':base_mae,
         'mae_improvement_pct':100*(base_mae-mae)/base_mae,'val_mse_scaled':float(mean_squared_error(va['y'],p_val)),
         'directional_accuracy_pct':float(100*np.mean(np.sign(delta_pred)==np.sign(delta_true))),
         'delta_pred_std_raw':float(np.std(delta_pred)), 'delta_true_std_raw':float(np.std(delta_true)),
         'delta_pred_mean_abs_raw':float(np.mean(np.abs(delta_pred))),
         'pred_delta_corr':float(np.corrcoef(delta_pred,delta_true)[0,1]) if np.std(delta_pred)>1e-12 else np.nan,
         'test_start':str(pd.Timestamp(te['date'][0]).date()),'fit_start':str(data.Date.iloc[ctx['fit_start']].date())}
    if extras: row.update(extras)
    predictions=pd.DataFrame({'Date':te['date'],'ratio':ctx['ratio'],'scheme':ctx['scheme'],
       'feature_set':ctx['feature_set'],'model':model_name,'seed':int(seed),
       'actual':y_true,'pred':pred,'naive':base,'pred_delta':delta_pred,'actual_delta':delta_true})
    return row,predictions

def train_one(ctx,name,seed=SEED, epochs=EPOCHS,patience=PATIENCE):
    tr,va,te=(ctx[k] for k in ('train','val','test'))
    if name=='naive':
        return va['prev'],te['prev'],{'epochs_run':0,'fit_seconds':0,'selected_params':'none'}
    residual=name.endswith('_residual')
    if name=='ridge_residual':
        model=Ridge(alpha=1.0)
        t0=time.monotonic()
        model.fit(tr['X'].reshape(len(tr['X']),-1),tr['y']-tr['prev'],
                  sample_weight=sample_weights(len(tr['idx']),ctx['scheme']))
        pv=va['prev']+model.predict(va['X'].reshape(len(va['X']),-1))
        pt=te['prev']+model.predict(te['X'].reshape(len(te['X']),-1))
        return pv,pt,{'epochs_run':0,'fit_seconds':time.monotonic()-t0,'selected_params':'Ridge alpha=1'}
    arch=name.rsplit('_',1)[0]
    tf.keras.backend.clear_session();gc.collect();tf.keras.utils.set_random_seed(seed)
    model=build_net(arch,tr['X'].shape[1:])
    target_tr=tr['y']-tr['prev'] if residual else tr['y']
    target_va=va['y']-va['prev'] if residual else va['y']
    cb=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=patience,restore_best_weights=True)]
    start=time.monotonic()
    history=model.fit(tr['X'],target_tr,validation_data=(va['X'],target_va),
            sample_weight=sample_weights(len(tr['idx']),ctx['scheme']),epochs=epochs,
            batch_size=BATCH_SIZE,shuffle=False,verbose=0,callbacks=cb)
    pv=model.predict(va['X'],verbose=0).ravel();pt=model.predict(te['X'],verbose=0).ravel()
    if residual:pv=va['prev']+pv;pt=te['prev']+pt
    extra={'epochs_run':len(history.history['loss']),'fit_seconds':time.monotonic()-start,
           'selected_params':f'{arch}, units fixed, lr=.001'}
    del model;tf.keras.backend.clear_session();gc.collect()
    return pv,pt,extra

# A semantic audit: if all residual delta predictions=0, result is exactly Naive.
assert np.allclose(np.array([.2,.7])+np.zeros(2),np.array([.2,.7]))

## 6. Full factorial — all models × all feature groups × all strategies × all splits

**Quan trọng:** Tất cả 9 fitted models được chạy trên mỗi combination; không giới hạn chiến lược cho Paper3 như V17. Checkpoint CSV được ghi sau **từng fit**; rerun bỏ qua tổ hợp đã có. Với `FULL_MODE`, chạy notebook đến khi count đạt expected trước khi báo cáo.

**Chú ý:** Đây là *single split* per ratio; `expanding/rolling/recency` ở grid là cách chọn training data trước test, **không phải walk-forward refit**. Walk-forward nằm ở Cell riêng.

In [ ]:
RESULTS_FILE=OUT/'grid_results.csv';PREDS_FILE=OUT/'grid_predictions.csv'
GRID_COLS=['ratio','feature_set','scheme','model','seed']

def load_csv(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()

def completed_keys():
    existing=load_csv(RESULTS_FILE)
    if existing.empty:return set()
    return set((round(float(r.ratio),5),str(r.feature_set),str(r.scheme),str(r.model),int(r.seed))
               for r in existing.itertuples(index=False))

def save_incremental(row,preds):
    pd.DataFrame([row]).to_csv(RESULTS_FILE,mode='a',header=not RESULTS_FILE.exists(),index=False)
    preds.to_csv(PREDS_FILE,mode='a',header=not PREDS_FILE.exists(),index=False)

if RUN_FULL_FACTORIAL:
    done=completed_keys()
    expected=len(RATIOS)*len(FEATURE_SETS)*len(SCHEMES)*(len(MODEL_NAMES)+1)
    print('\n' + '═'*80)
    print(f'FULL GRID | {len(done)}/{expected} completed (resume-safe)')
    print('═'*80, flush=True)
    for ratio in RATIOS:
      ratio_jobs = len(FEATURE_SETS)*len(SCHEMES)*(len(MODEL_NAMES)+1)
      print(f'\n┏━━ SPLIT {ratio*100:g}/{(1-ratio)*100:g}  | {ratio_jobs} experiments ━━┓',flush=True)
      for feature_set in FEATURE_SETS:
        print(f'\n  FEATURE SET: {feature_set} | {", ".join(FEATURE_SETS[feature_set])}',flush=True)
        for scheme in SCHEMES:
          ctx=make_context(data,ratio,scheme,feature_set)
          print(f'  ── {scheme:<18} | train={len(ctx["train"]["idx"]):,} '
                f'val={len(ctx["val"]["idx"]):,} test={len(ctx["test"]["idx"]):,} ──',flush=True)
          for name in ['naive']+MODEL_NAMES:
            key=(round(ratio,5),feature_set,scheme,name,SEED)
            if key in done:
                print(f'     ↳ {name:<21} [SKIP] checkpoint exists',flush=True)
                continue
            job_start=time.monotonic()
            print(f'     ▶ {len(done)+1:>3}/{expected}  {name:<21} training...',end='',flush=True)
            pv,pt,extra=train_one(ctx,name,seed=SEED)
            row,pr=evaluate(ctx,name,pv,pt,seed=SEED,extras=extra)
            save_incremental(row,pr)
            done.add(key)
            duration=time.monotonic()-job_start
            print(f' done | MAE={row["mae"]:.5f} | MSE={row["mse"]:.6f} | RMSE={row["rmse"]:.5f} '
                  f'| vs Naive={row["mae_improvement_pct"]:+.2f}% '
                  f'| epochs={int(extra.get("epochs_run",0)):>2} | {duration:.1f}s',flush=True)
      print(f'  ┗━━ Completed split {ratio*100:g}/{(1-ratio)*100:g} ━━┛',flush=True)
    assert len(done)>=expected, f'Full grid incomplete: {len(done)}/{expected}'
    print('\n' + '═'*80)
    print(f'FULL GRID COMPLETE: {len(done)} / {expected} jobs ✓')
    print('═'*80)
else:
    print('Skipped full grid by configuration')

## 7. Fair-comparison tables, paired metrics, residual audit, Actual vs Predicted

Không xếp hạng các ratio khác nhau bằng raw MAE vì test periods khác nhau. Chọn `best` chỉ để mô tả **sau đánh giá**; không dùng test để retrain hoặc chọn hyperparameters.

In [ ]:
grid=load_csv(RESULTS_FILE)
if grid.empty: raise RuntimeError('Chưa có kết quả grid; chạy Cell 6 trước.')
expected=len(RATIOS)*len(FEATURE_SETS)*len(SCHEMES)*(len(MODEL_NAMES)+1)
print('\n'+'═'*80)
print(f'FAIR COMPARISON | Completed {len(grid)}/{expected} jobs')
print('═'*80)
if len(grid)<expected: print('WARNING: INCOMPLETE GRID. Không báo cáo bảng này là kết luận cuối.')
for ratio in RATIOS:
    tab=grid[np.isclose(grid.ratio,ratio)].copy()
    tab=tab.sort_values(['feature_set','scheme','mae'])
    print(f'\n━━ TRAIN/TEST {ratio*100:g}/{(1-ratio)*100:g} | test={tab.n_test.iloc[0]:,} phiên ━━');display(tab[['feature_set','scheme','model','mae','mse','rmse','mae_improvement_pct','directional_accuracy_pct','val_mse_scaled']].round(5))
    tab.to_csv(OUT/f'fair_table_{ratio:.3f}.csv',index=False)
# All model predictions MUST match corresponding Naive dates and labels
pr=load_csv(PREDS_FILE);pr['Date']=pd.to_datetime(pr['Date'])
for (ratio,feature,scheme),part in pr.groupby(['ratio','feature_set','scheme']):
    ref=part[part.model=='naive'].sort_values('Date')
    if ref.empty:continue
    for name,g in part.groupby('model'):
        g=g.sort_values('Date')
        assert np.array_equal(ref.Date.to_numpy(),g.Date.to_numpy()),('Dates mismatch',ratio,feature,scheme,name)
        assert np.allclose(ref.actual,g.actual,atol=1e-5),('Actual mismatch',ratio,feature,scheme,name)
print('PASS: matched timestamps and same raw test target within all runs')
# Residual audit: predicted delta vs actual delta and naive
residual_rows=grid[grid.model.str.endswith('_residual')].copy()
display(residual_rows[['ratio','feature_set','scheme','model','mae','mse','rmse','mae_naive','mae_improvement_pct',
                       'delta_pred_std_raw','delta_true_std_raw','delta_pred_mean_abs_raw','pred_delta_corr']].round(5))
residual_rows.to_csv(OUT/'residual_audit.csv',index=False)
# Example plots: a priori choose Residual GRU and Naive for three feature sets, all ratios
for ratio in RATIOS:
  for feature in FEATURE_SETS:
    s=pr[(np.isclose(pr.ratio,ratio))&(pr.feature_set==feature)&(pr.scheme=='expanding')]
    s=s[s.model.isin(['naive','gru_residual','ridge_residual'])]
    fig,ax=plt.subplots(figsize=(13,4))
    for name,g in s.groupby('model'):
      g=g.sort_values('Date');ax.plot(g.Date,g.pred,label=name,linewidth=1.2 if name!='naive' else .9)
    ax.plot(s[s.model=='naive'].Date,s[s.model=='naive'].actual,label='Actual',c='black',lw=1.8)
    ax.set_title(f'Actual vs Predicted | ratio {ratio} | {feature} | expanding, horizon 1')
    ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.savefig(OUT/f'actual_pred_{ratio}_{feature}.png',dpi=140);plt.show()

## 8. Feature ablation: Copper-only vs Copper+Paper3 vs Copper+Legacy14

Chỉ so sánh cùng `(ratio, scheme, model, seed, dates)`; giữ Copper trong cả ba bộ feature. Bảng cho biết **giá trị bổ sung** của các biến ngoại sinh, không phải kết luận nhân quả.

In [ ]:
ab=grid[grid.model!='naive'].pivot_table(index=['ratio','scheme','model','seed'],columns='feature_set',values='mae',aggfunc='first')
for key in FEATURE_SETS:
    if key not in ab.columns: ab[key]=np.nan
ab['paper3_gain_pct']=100*(ab['copper_only']-ab['copper_paper3'])/ab['copper_only']
ab['legacy14_gain_pct']=100*(ab['copper_only']-ab['copper_legacy14'])/ab['copper_only']
ab=ab.reset_index();ab.to_csv(OUT/'feature_ablation.csv',index=False)
display(ab.round(4))
print('Positive gain means foreign features improved MAE relative to Copper-only.')

## 9. Walk-forward: Residual-GRU expanding vs Naive, five seeds, paired DM (HAC)

**Protocol:** tách các mốc bắt đầu từ năm 2018, retrain Expanding GRU ở mỗi fold chỉ bằng lịch sử trước fold; tính validation bên trong historical train; dự báo next-day trong block test, **không train trên nhãn future block**. 5 seeds, step=63 phiên trong FULL (chi phí lớn). Cùng forecast dates với Naive.

**Thống kê:** Với mỗi seed, loss differential `|err_GRU|-|err_Naive|`, HAC-normal approximate p-value. Vì 5 seeds dùng cùng test dates, **không** coi 5 lần chạy như 5 bộ mẫu độc lập để phóng đại significance. Các bài báo nghiêm ngặt cần xem xét điều chỉnh kiểm định khi chọn nhiều mô hình.

In [ ]:
WF_ROWS=OUT/'walk_forward_fold_metrics.csv'; WF_PREDS=OUT/'walk_forward_predictions.csv'

def make_fold_context(frame, fold_start,fold_end,feature_set):
    # Fold temporally extends training only to fold_start. Fit sc on historical subtrain before internal val.
    val_start=int(fold_start*.9)
    assert val_start>LOOKBACK+25 and fold_start<len(frame)
    cols=FEATURE_SETS[feature_set]
    xsc=MinMaxScaler().fit(frame.iloc[:val_start][cols]);ysc=MinMaxScaler().fit(frame.iloc[:val_start][['Copper']])
    x=xsc.transform(frame[cols]).astype('float32');y=ysc.transform(frame[['Copper']]).ravel().astype('float32')
    prev=np.r_[np.nan,y[:-1]]
    def make(s,e):
        ids=np.arange(max(s,LOOKBACK),e,dtype=int)
        return {'idx':ids,'X':np.stack([x[t-LOOKBACK:t] for t in ids]),
                'y':y[ids],'prev':prev[ids],'date':frame.Date.iloc[ids].to_numpy()}
    tr=make(LOOKBACK,val_start);va=make(val_start,fold_start);te=make(fold_start,fold_end)
    assert tr['idx'][-1]<va['idx'][0]<te['idx'][0] and te['idx'][-1]<len(frame)
    assert np.allclose(te['prev'],te['X'][:,-1,0],atol=2e-6)
    return {'train':tr,'val':va,'test':te,'ysc':ysc,'xsc':xsc,'features':cols,'feature_set':feature_set,
            'scheme':'expanding','ratio':float(fold_start/len(frame)),'fit_start':0,'val_start':val_start,'test_start':fold_start}

if RUN_WALK_FORWARD:
    starts=np.flatnonzero(data.Date.dt.year.to_numpy()>=WF_START_YEAR)
    if len(starts)==0: raise ValueError(f'Dataset không có năm >= {WF_START_YEAR}')
    wf_first=int(starts[0]);folds=[(i,min(i+WF_STEP,len(data))) for i in range(wf_first,len(data),WF_STEP)]
    existing=load_csv(WF_ROWS)
    done=set((int(r.fold_start),int(r.seed)) for r in existing.itertuples(index=False)) if not existing.empty else set()
    print('\n'+'═'*80)
    print(f'WALK-FORWARD | {len(folds)} folds × {len(WF_SEEDS)} seeds = {len(folds)*len(WF_SEEDS)} fits')
    print(f'Completed checkpoints: {len(done)} | Feature set: {WF_FEATURE_SET}')
    print('═'*80)
    for start,end in folds:
      ctx=make_fold_context(data,start,end,WF_FEATURE_SET)
      for seed in WF_SEEDS:
        if (start,seed) in done:continue
        print(f'▶ WF {len(done)+1:>3}/{len(folds)*len(WF_SEEDS)} | {data.Date.iloc[start].date()} '
              f'→ {data.Date.iloc[end-1].date()} | seed={seed}',end='',flush=True)
        pv,pt,extra=train_one(ctx,'gru_residual',seed,epochs=WF_EPOCHS,patience=WF_PATIENCE)
        row,frame=evaluate(ctx,'gru_residual',pv,pt,seed=seed,extras=extra)
        row['fold_start']=start;row['fold_end']=end
        pd.DataFrame([row]).to_csv(WF_ROWS,mode='a',header=not WF_ROWS.exists(),index=False)
        frame['fold_start']=start;frame['fold_end']=end
        frame.to_csv(WF_PREDS,mode='a',header=not WF_PREDS.exists(),index=False)
        done.add((start,seed))
        print(f' | MAE={row["mae"]:.5f} | MSE={row["mse"]:.6f} | RMSE={row["rmse"]:.5f} | Naive={row["mae_naive"]:.5f} '
              f'| gain={row["mae_improvement_pct"]:+.2f}% | {extra["fit_seconds"]:.1f}s',flush=True)
    assert len(done)>=len(folds)*len(WF_SEEDS), f'Incomplete WF: {len(done)}'
print('Walk-forward saved:',WF_PREDS)

## 10. Statistical evaluation: DM-style HAC and year-by-year error

DM-style HAC uses Newey–West long-run variance. `positive mean diff` means **GRU worse than Naive**. For overlapping horizons, choose HAC lags carefully; horizon 1 and refitting every 63 days here. Report both p-value and effect sizes, **do not call 0.05 definitive truth**.

In [ ]:
def hac_test(d,max_lags=5):
    d=np.asarray(d,dtype=float);d=d[np.isfinite(d)];n=len(d)
    if n<20:return {'n':n,'mean_difference':np.nan,'z':np.nan,'p_value_two_sided':np.nan}
    d0=d-d.mean();L=min(max_lags,n-2)
    var=np.dot(d0,d0)/n
    for k in range(1,L+1):
        gamma=np.dot(d0[k:],d0[:-k])/n
        var+=2*(1-k/(L+1))*gamma
    se=np.sqrt(max(var,1e-18)/n)
    z=d.mean()/se
    return {'n':n,'mean_difference':float(d.mean()),'z':float(z),'p_value_two_sided':float(2*norm.sf(abs(z)))}

wf=load_csv(WF_PREDS)
if wf.empty: print('No walk-forward results yet.')
else:
    wf['Date']=pd.to_datetime(wf['Date'])
    tests=[];yearly=[]
    for seed,part in wf.groupby('seed'):
        part=part.sort_values('Date')
        assert not part.Date.duplicated().any(), 'WF overlapping dates'
        d=np.abs(part.actual-part.pred)-np.abs(part.actual-part.naive)
        test=hac_test(d,5)
        test.update({'seed':int(seed),'mae_gru':float(np.mean(np.abs(part.actual-part.pred))),
                     'mse_gru':float(np.mean((part.actual-part.pred)**2)),
                     'mse_naive':float(np.mean((part.actual-part.naive)**2)),
                     'mae_naive':float(np.mean(np.abs(part.actual-part.naive))),
                     'directional_accuracy_pct':float(100*np.mean(np.sign(part.pred_delta)==np.sign(part.actual_delta)))})
        tests.append(test)
        for year,g in part.groupby(part.Date.dt.year):
            yearly.append({'seed':int(seed),'year':int(year),'n':len(g),
                          'mae_gru':float(np.mean(np.abs(g.actual-g.pred))),
                          'mae_naive':float(np.mean(np.abs(g.actual-g.naive)))})
    testtable=pd.DataFrame(tests);yeartable=pd.DataFrame(yearly)
    testtable.to_csv(OUT/'dm_hac_by_seed.csv',index=False)
    yeartable.to_csv(OUT/'wf_by_year.csv',index=False)
    display(testtable.round(5));display(yeartable.groupby('year')[['mae_gru','mae_naive']].agg(['mean','std']).round(5))
    print('Across seeds (descriptive, NOT independent experiments):')
    display(testtable[['mae_gru','mae_naive','mse_gru','mse_naive','directional_accuracy_pct']].agg(['mean','std']).round(5))
    p=wf[wf.seed==min(wf.seed)].sort_values('Date')
    fig,ax=plt.subplots(figsize=(14,4));ax.plot(p.Date,p.actual,label='Actual',lw=1.4,color='black');ax.plot(p.Date,p.pred,label='Residual-GRU',alpha=.75);ax.plot(p.Date,p.naive,label='Naive',alpha=.65)
    ax.legend();ax.grid(alpha=.2);ax.set_title('Walk-forward: Actual vs Residual-GRU vs Naive (one seed)')
    plt.tight_layout();plt.savefig(OUT/'wf_actual_vs_pred.png',dpi=150);plt.show()
    fig,ax=plt.subplots(figsize=(14,3));ax.plot(p.Date,p.pred_delta,label='Predicted delta',alpha=.8);ax.plot(p.Date,p.actual_delta,label='Actual delta',alpha=.7)
    ax.axhline(0,c='black',lw=.7);ax.legend();ax.grid(alpha=.2);ax.set_title('Residual audit: does GRU learn nonzero price movements?')
    plt.tight_layout();plt.savefig(OUT/'wf_delta_audit.png',dpi=150);plt.show()

## 11. Experiment manifest & export ZIP

File ZIP gồm **raw prediction timestamps**, audit, bảng metrics, model recipes, cấu hình và charts. Để báo cáo, FULL grid phải đủ số jobs và WF đủ seeds × folds. Không ngầm gộp QUICK và FULL.

In [ ]:
import platform, sklearn
manifest={
 'dataset':str(data_path),'dataset_sha256':hashlib.sha256(Path(data_path).read_bytes()).hexdigest(),
 'n_rows_clean':int(len(data)),'date_start':str(data.Date.iloc[0].date()),'date_end':str(data.Date.iloc[-1].date()),
 'mode':MODE,'lookback':LOOKBACK,'horizon':HORIZON,'ratios':RATIOS,'feature_sets':FEATURE_SETS,
 'strategies':SCHEMES,'neural_architectures':NN_ARCHES,'models':MODEL_NAMES,
 'epochs_full_grid':EPOCHS,'patience_full_grid':PATIENCE,'seed_full_grid':SEED,
 'wf_feature_set':WF_FEATURE_SET,'wf_start_year':WF_START_YEAR,'wf_step':WF_STEP,
 'wf_seeds':WF_SEEDS,'wf_epochs':WF_EPOCHS,
 'versions':{'python':platform.python_version(),'tensorflow':tf.__version__,'numpy':np.__version__,
             'pandas':pd.__version__,'scikit_learn':sklearn.__version__},
 'warnings':['CSV does not verify actual information publication times across markets',
             'Within grid, train/validation fit is fixed; rolling is not repeated refit',
             'Validation is not reintroduced into final train after tuning',
             'Feature sets are a priori; Spearman shown only for EDA, not test-dependent selection',
             'Test comparison of many models is exploratory; no test-based tuning']}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
output_zip=OUT.parent/f'Copper_NonSA_Fair_V20_{RUN_KEY}_results.zip'
with zipfile.ZipFile(output_zip,'w',zipfile.ZIP_DEFLATED) as z:
  for p in OUT.glob('*'):
    if p.is_file():z.write(p,arcname=p.name)
print('ZIP:',output_zip)
print('Grid:',len(load_csv(RESULTS_FILE)),'expected:',len(RATIOS)*len(FEATURE_SETS)*len(SCHEMES)*(len(MODEL_NAMES)+1))
print('WF:',len(load_csv(WF_ROWS)),'fold-seed fits')

## 12. Checklist nghiên cứu trước khi kết luận

- [ ] Kiểm tra dữ liệu nguồn (đặc biệt **giờ đóng cửa** của các thị trường), missing, anomalies và ngày giao dịch.
- [ ] Kiểm tra scaler từng split và assert đầu vào Copper(t−1), không Copper(t).
- [ ] Chạy FULL grid **360 lượt (324 fits + 36 Naive)**, không tính phần walk-forward, hoàn thành resume.
- [ ] Đọc ablation Copper-only vs Copper+Paper3 vs Copper+Legacy14 trên cùng timestamps.
- [ ] Đọc residual audit: `delta_pred_std`, `delta_pred_mean_abs`, correlation delta và DA.
- [ ] Chạy walk-forward Residual-GRU Expanding + 5 seeds + DM/HAC; giải thích CI/variance, không chọn mô hình theo test.
- [ ] Tổng hợp 4 ratios 80/20, 90/10, 92.5/7.5, 95/5 với figures và hạn chế so sánh giữa test periods.
- [ ] Nêu rõ nhóm *reproduction* bài báo gốc (không gồm Copper lịch sử) và nhóm *autoregressive extension* (có Copper lịch sử) — không xem là reproduction chính xác.
- [ ] Không tuyên bố hiệu năng cải thiện nếu không thắng Naive qua nhiều giai đoạn và/hoặc không có ý nghĩa thống kê.

**Technical nuance:** `Copper+Paper3` và `Copper+Legacy14` dùng nhiều đặc trưng hơn `Copper-only`, nhưng Ridge sử dụng đúng 22×features đã flatten, không thêm feature engineering ưu đãi. Có thể thêm engineered features trong **nghiên cứu khác**, không trộn vào bảng công bằng.